# Fixed-point bundle method: quickstart

This notebook builds one operator on the product-of-simplex domain $\Delta_n^m$ with
user-selected dimensions `m, n`. The same construction covers the simplex case: at `m = 1`
the operator reduces to an operator on $\Delta_n$. Both cases are solved by the solver `FP_VI`; the notebook then plots the convergence trace.

**Repository note:** the setup cell checks the current directory, its parents, and a
`FiberBundle_VI` subdirectory for a local `solver/` package. If no local clone is found
(for example in Colab), it clones the repository.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repository(start):
    candidates = [start, *start.parents, start / "FiberBundle_VI"]
    for path in candidates:
        if (path / "solver").is_dir() and (path / "main.py").is_file():
            return path.resolve()
    return None


root = find_repository(Path.cwd())
if root is None:
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/shb20tsinghua/FiberBundle_VI.git"],
        check=True,
    )
    root = (Path.cwd() / "FiberBundle_VI").resolve()

os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt

from solver import FP_VI
from builtin_model import DerivFreeWrapper, nn_F, payoff_F

print("repository:", root)
print("Python     ", sys.version.split()[0])
print("NumPy      ", np.__version__)
print("SciPy      ", scipy.__version__)
print("matplotlib ", matplotlib.__version__)

if not hasattr(np, "matvec"):
    np.matvec = lambda matrix, vector: np.matmul(matrix, vector[..., None])[..., 0]

## Custom operator

The solver needs an object with a `value_Jaco(x) -> (F, JF)` method. This demo uses

$$
F(x)=b+A x+\tfrac12 x^{\circ 2},
\qquad
JF(x)=A+\operatorname{diag}(x),
$$

where the square is elementwise and `x` is flattened internally.

`DerivFreeWrapper` instead needs only a `value(x) -> F` method, so the demo below
implements both methods.


In [ ]:
class CustomOperator:
    """F(x) = b + A x + x**2 / 2, with x flattened internally."""

    def __init__(self, A, b):
        self.A, self.b = A, b

    def value(self, x):  # implement this for derivative-free mode
        shape = x.shape
        flat = x.ravel()
        values = self.b + self.A @ flat + 0.5 * flat**2
        return values.reshape(shape)

    def value_Jaco(self, x):  # implement this for derivative-based mode
        shape = x.shape
        flat = x.ravel()
        values = self.b + self.A @ flat + 0.5 * flat**2
        jacobian = self.A.copy()
        jacobian[np.diag_indices_from(jacobian)] += flat
        return values.reshape(shape), jacobian.reshape(shape + shape)


## Build operator instance

The settings cell chooses `m, n`, the operator and the solution mode:

- `choice = 0`: the quadratic custom operator defined above;
- `choice = 1`: the built-in three-layer neural network `nn_F` with widths `[m*n, m*n, m*n]`;
- `choice = 2`: the built-in low-rank payoff-tensor operator `payoff_F` of rank `m*n`.

`derivfree` chooses whether to solve in derivative-free mode. `nn_F`, `payoff_F`, and
`CustomOperator` all provide both `value` and `value_Jaco`. A custom operator only needs to implement the method that is actually
used: `value` for derivative-free mode, or `value_Jaco` for derivative-based mode.
Derivative-free mode is recommended, since the experiments show that it needs nearly the
same number of iterations as derivative-based mode.


In [ ]:
m, n = 16, 16
SEED = 0
choice = 0
derivfree = True

rng = np.random.default_rng(SEED)

if choice == 0:
    A = rng.standard_normal((m * n, m * n))
    b = rng.standard_normal(m * n)
    VI_to_solve = CustomOperator(A, b)
elif choice == 1:
    VI_to_solve = nn_F(SEED, [m * n, m * n, m * n], 0)
else:
    VI_to_solve = payoff_F(SEED, m, n, m * n)
VI_to_solve = DerivFreeWrapper(VI_to_solve) if derivfree else VI_to_solve

## Solve

The solver `FP_VI` takes a starting point of shape `(m, n)` and returns the solution together with
`info = [nit, gap, barrier_sum, G_norm, avoid_n]`.

`GAP_TOL = 1e-8` specifies the required precision. Passing `verbose=True`
prints the per-iteration trace; the next cell captures the printed output for
the plot.


In [ ]:
import contextlib
import io

GAP_TOL = 1e-8

x_init = rng.dirichlet(np.ones(n), size=(m,))
trace_buffer = io.StringIO()
with contextlib.redirect_stdout(trace_buffer):
    solution_x, info = FP_VI(VI_to_solve).solve(
        x_init, gap_tol=GAP_TOL, verbose=True
    )

trace = trace_buffer.getvalue()
print(f"recorded {len(trace.splitlines()) - 1} iterations")
print(info)


## Convergence

The plot shows the gap on a logarithmic scale. The solver stops once the gap is below
`GAP_TOL`. The title reflects the selected domain.

In [ ]:
rows = [line.split("|") for line in trace.splitlines()[1:]]
nits = np.array([int(row[1]) for row in rows])
gaps = np.array([float(row[2]) for row in rows])

fig, ax = plt.subplots(figsize=(6, 4), dpi=140)
ax.semilogy(nits, gaps)
ax.set_title(
    f"Simplex VI (dim={n})" if m == 1
    else f"Product-of-simplex VI (dim={m} × {n})"
)
ax.set_xlabel("iteration")
ax.set_ylabel("gap")
ax.grid(True, which="both", alpha=0.25)
plt.tight_layout()
plt.show()